### Построение базовой модели

In [9]:
import numpy as np
import json

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

np.random.seed = 42

In [3]:
with open('./data/train_data.json', 'r', encoding='utf-8') as file:
    train_data = json.load(file)

X_train = train_data['X']
y_train = train_data['y']

print(len(X_train), len(y_train))

with open('./data/test_data.json', 'r', encoding='utf-8') as file:
    test_data = json.load(file)

X_test = test_data['X']
y_test = test_data['y']

print(len(X_test), len(y_test))

25000 25000
25000 25000


In [4]:
tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',    # нормализация символов
    ngram_range=(1, 2),         # униграммы + биграммы
    min_df=2,                   # отсечение очень редкие токены
    max_df=0.95,                # отсекчение слишком частые токены
    max_features=50000,         # ограничение размерности
    sublinear_tf=True
)

pipe = Pipeline([
    ('tfidf', tfidf),
    ('clf', LinearSVC())
])

param_grid = [

    # LinearSVC
    {
        'clf': [LinearSVC()],
        'clf__C': [0.1, 0.5, 1, 2],
        'tfidf__ngram_range': [(1,1), (1,2)],
        'tfidf__min_df': [1, 2, 3],
        'tfidf__max_df': [0.9, 0.95],
        'tfidf__max_features': [30000, 50000, 70000, 100000] 
    },

    # Logistic Regression
    {
        'clf': [LogisticRegression(max_iter=2000, solver='liblinear')],
        'clf__C': [0.1, 0.5, 1, 2],
        'tfidf__ngram_range': [(1,1), (1,2)],
        'tfidf__min_df': [1, 2, 3],
        'tfidf__max_df': [0.9, 0.95],
        'tfidf__max_features': [30000, 50000, 70000, 100000] 
    }
]

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print('\nЛучшие параметры:')
print(grid.best_params_)

print('\nЛучший CV score:')
print(grid.best_score_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print('\nTest Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, digits=4))

Fitting 3 folds for each of 384 candidates, totalling 1152 fits

Лучшие параметры:
{'clf': LinearSVC(), 'clf__C': 0.1, 'tfidf__max_df': 0.95, 'tfidf__max_features': 30000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2)}

Лучший CV score:
0.8822000396214552

Test Accuracy: 0.90188

Classification Report:
              precision    recall  f1-score   support

           0     0.9066    0.8961    0.9013     12500
           1     0.8973    0.9077    0.9024     12500

    accuracy                         0.9019     25000
   macro avg     0.9019    0.9019    0.9019     25000
weighted avg     0.9019    0.9019    0.9019     25000



#### После проведения тестов, лучшей базовой моделью оказалась LinearSVC с параметрами: 

- C = 0.1

#### и TfIdf векторизатором с параметрами:

- ngram_range = (1, 2)

- max_df = 0.9

- min_df = 1

- max_features = 30000

### Построение CNN модели

In [11]:
import re

from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [12]:
_token_re = re.compile(r"[A-Za-z']+")

def tokenize(text: str):
    # простая англ. токенизация: слова и апострофы
    return _token_re.findall(text.lower())

PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'

def build_vocab(texts, min_freq=2, max_size=50000):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    # самые частые слова в словарь, но частота их должна быть больше min_freq
    most_common = [w for w, c in counter.most_common() if c >= min_freq]
    # обрезаем словарь частых слов по максимальной длине словаря max_size
    if max_size is not None:
        most_common = most_common[:max_size]
    
    # создаем словарь индексов
    stoi = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    # присваиваем каждому слову уникальный номер
    for w in most_common:
        if w not in stoi:
            stoi[w] = len(stoi)
    return stoi

def encode(text, stoi, max_len):
    tokens = tokenize(text)
    # перевод слов в числа с помощью словаря stoi
    ids = [stoi.get(w, stoi[UNK_TOKEN]) for w in tokens]
    # обрезаем если отзыв слишком длинный
    ids = ids[:max_len]
    
    # если отзыв короткий, добавляем нули
    if len(ids) < max_len:
        ids += [stoi[PAD_TOKEN]] * (max_len - len(ids))

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, stoi, max_len):
        self.texts = texts
        self.labels = labels
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, index):
        X = torch.tensor(encode(self.texts[index], self.stoi, self.max_len), dtype=torch.long)
        y = torch.tensor(self.labels[index], dtype=torch.long)
        return X, y
    
class TextCNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        num_classes: int = 2,
        kernel_sizes=(3, 4, 5),
        num_filters: int = 128,
        dropout: float = 0.5,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=ks)
            for ks in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc =nn.Linear(num_filters * len(kernel_sizes), num_classes)